####  INICIAR NOTEBOOK


In [1]:
%idle_timeout 60
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job

from pyspark.context import SparkContext
from pyspark.sql import functions as F

import re
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 60 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 60
Session ID: 6fd5f605-1d8e-4c7a-acad-19743fad4ae8
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 6fd5f605-1d8e-4c7a-acad-19743fad4ae8 to get into ready status...
Session 6fd5f605-1d8e-4c7a-acad-19743fad4ae8 has 

#### 1.0 Declarar o bucket e funções necessárias


In [2]:
BUCKET = "fiap26-data-analytics-824232672586"

Função para criar as tabelas fato, consolida as respostas hoje colunadas em linhas para facilitar consulta

In [17]:
def unpivot_binarias(df, id_col, ano, prefixo, categoria, rotulos=None):
    if rotulos:
        indicadoras = [c for c in rotulos if c in df.columns]
    else:
        indicadoras = [
            c for c in df.columns
            if re.match(rf"^{re.escape(prefixo)}\d+_", c)
            and "nao_utilizo" not in c and "nenhuma" not in c
        ]
    if not indicadoras:
        print(f"AVISO ({ano}, {categoria}): nenhuma coluna encontrada com prefixo '{prefixo}'")
        return None

    def rotulo(c):
        if rotulos and c in rotulos:
            return rotulos[c]
        sem_prefixo = re.sub(rf"^{re.escape(prefixo)}\d+_", "", c)
        return sem_prefixo.replace("_", " ").title()

    n = len(indicadoras)
    pares = ", ".join(f"'{rotulo(c)}', `{c}`" for c in indicadoras)
    stack_expr = f"stack({n}, {pares}) as (tecnologia, utilizado)"
    return (df.selectExpr(id_col, f"cast({ano} as int) as ano_pesquisa", stack_expr)
              .filter("utilizado = '1'")
              .withColumn("categoria", F.lit(categoria))
              .select(F.col(id_col).alias("respondente_id"), "ano_pesquisa", "categoria", "tecnologia"))

#### 2.0 Ler camada silver


In [3]:
silver_2023 = glueContext.create_dynamic_frame.from_catalog(database="silver_db", table_name="tbl_state_of_data_2023").toDF()
silver_2024 = glueContext.create_dynamic_frame.from_catalog(database="silver_db", table_name="tbl_state_of_data_2024").toDF()
silver_2025 = glueContext.create_dynamic_frame.from_catalog(database="silver_db", table_name="tbl_state_of_data_2025").toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


#### 3.0 Criação das tabelas fato


##### 3.1 Fato Perfil Profissional

In [4]:
CAMPOS_PADRAO = [
    "respondente_id", "idade", "faixa_idade", "genero", "cor_raca_etnia", "pcd",
    "uf_onde_mora", "regiao_onde_mora", "nivel_ensino", "setor", "numero_funcionarios",
    "atua_como_gestor", "cargo_atual", "senioridade", "faixa_salarial",
    "modelo_trabalho_atual", "cloud_preferida", "usa_chatgpt_llm_pessoal",
    "ai_generativa_prioridade_empresa", "ano_pesquisa",
]

def so_campos_padrao(df, ano):
    cols = [c for c in CAMPOS_PADRAO if c in df.columns]
    faltando = [c for c in CAMPOS_PADRAO if c not in df.columns]
    if faltando:
        print(f"AVISO ({ano}): campos padrao ausentes na Silver: {faltando}")
    return df.select(*cols)

gold_core = (so_campos_padrao(silver_2023, 2023)
             .unionByName(so_campos_padrao(silver_2024, 2024), allowMissingColumns=True)
             .unionByName(so_campos_padrao(silver_2025, 2025), allowMissingColumns=True))

gold_core.groupBy("ano_pesquisa").count().show()

+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|        2023| 5293|
|        2024| 5215|
|        2025| 3494|
+------------+-----+


In [23]:
dyf_core = DynamicFrame.fromDF(gold_core, glueContext, "gold_core")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_perfil_profissional/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_perfil_profissional")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_core)

##### 3.2 Fato Tecnologia


In [18]:
ROTULOS_LINGUAGEM_2023 = {
    "p4_d_1_sql": "SQL", "p4_d_2_r": "R", "p4_d_3_python": "Python",
    "p4_d_4_c_c_c": "C/C++/C#", "p4_d_5_net": ".NET", "p4_d_6_java": "Java",
    "p4_d_7_julia": "Julia", "p4_d_8_sas_stata": "SAS/Stata",
    "p4_d_9_visual_basic_vba": "Visual Basic/VBA", "p4_d_10_scala": "Scala",
    "p4_d_11_matlab": "Matlab", "p4_d_12_rust": "Rust", "p4_d_13_php": "PHP",
    "p4_d_14_javascript": "JavaScript",
    # p4_d_15 = "Não utilizo nenhuma linguagem" -- excluída de propósito
}

ROTULOS_LINGUAGEM_2024 = {
    "4_d_1_sql": "SQL", "4_d_2_r": "R", "4_d_3_python": "Python",
    "4_d_4_c_c_c": "C/C++/C#", "4_d_5_net": ".NET", "4_d_6_java": "Java",
    "4_d_7_julia": "Julia", "4_d_8_sas_stata": "SAS/Stata",
    "4_d_9_visual_basic_vba": "Visual Basic/VBA", "4_d_10_scala": "Scala",
    "4_d_11_matlab": "Matlab", "4_d_12_rust": "Rust", "4_d_13_php": "PHP",
    "4_d_14_javascript": "JavaScript",
    # 4_d_15 = "Não utilizo nenhuma das linguagens listadas" -- excluída
}

ROTULOS_LINGUAGEM_2025 = {
    "4_c_1_sql": "SQL", "4_c_2_r": "R", "4_c_3_python": "Python",
    "4_c_4_c_c_c": "C/C++/C#", "4_c_5_julia": "Julia",
    "4_c_6_visual_basic_vba": "Visual Basic/VBA", "4_c_7_scala": "Scala",
    "4_c_8_dax": "DAX", "4_c_9_rust": "Rust",
    # 4_c_10 = "Não utilizo nenhuma das linguagens listadas" -- excluída
}

ling_2023 = unpivot_binarias(silver_2023, "respondente_id", 2023, "p4_d_", "linguagem", ROTULOS_LINGUAGEM_2023)
ling_2024 = unpivot_binarias(silver_2024, "respondente_id", 2024, "4_d_", "linguagem", ROTULOS_LINGUAGEM_2024)
ling_2025 = unpivot_binarias(silver_2025, "respondente_id", 2025, "4_c_", "linguagem", ROTULOS_LINGUAGEM_2025)

fact_tecnologias = ling_2023.unionByName(ling_2024).unionByName(ling_2025)
fact_tecnologias.groupBy("categoria", "tecnologia").count().orderBy(F.desc("count")).show(5)

+---------+----------+-----+
|categoria|tecnologia|count|
+---------+----------+-----+
|linguagem|       SQL| 8064|
|linguagem|    Python| 7687|
|linguagem|         R| 1087|
|linguagem|      Java|  646|
|linguagem|JavaScript|  505|
+---------+----------+-----+
only showing top 5 rows


In [19]:
dyf_tec = DynamicFrame.fromDF(fact_tecnologias, glueContext, "gold_tecnologias")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_tecnologias/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_tecnologias")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_tec)

##### 3.3 Fato IA Generativa

##### 3.4 Fato Desafios Gestão